[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fvalenzuelag/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/blob/Mistral/RA1/IL1.1/4-langchain_memory.ipynb)


# 4. LangChain Memory - Gestión de Contexto Conversacional

## Objetivos de Aprendizaje
- Comprender la importancia de la memoria en conversaciones con LLMs
- Implementar diferentes tipos de memoria con LangChain
- Gestionar el contexto de conversaciones largas
- Optimizar el uso de tokens con estrategias de memoria

## ¿Por qué es Importante la Memoria?

Los LLMs son **stateless** por naturaleza: no recuerdan conversaciones anteriores. La memoria permite:
- **Contexto conversacional**: Referirse a mensajes anteriores
- **Personalización**: Recordar preferencias del usuario
- **Continuidad**: Mantener hilos de conversación coherentes
- **Experiencia natural**: Conversaciones que se sienten humanas

## Tipos de Memoria en LangChain

1. **ConversationBufferMemory**: Mantiene todo el historial
2. **ConversationSummaryMemory**: Resume conversaciones largas
3. **ConversationBufferWindowMemory**: Mantiene solo los N mensajes más recientes
4. **ConversationSummaryBufferMemory**: Combina resumen + buffer reciente

In [1]:
# --- Instalación de dependencias (se ejecuta solo en Google Colab) ---
# En local no hace nada: usa `pip install -r requirements.txt` desde la raíz del repo.
import sys
if "google.colab" in sys.modules:
    !pip install -q langchain-openai python-dotenv


In [2]:
# --- Credenciales: funciona en local (.env) y en Google Colab (Secrets) ---
import os
try:
    from google.colab import userdata          # Colab: panel 🔑 Secrets
    # Solo LLM_API_KEY es obligatorio. Los demás son opcionales: defínelos como
    # Secrets únicamente si quieres usar otro proveedor o modelo.
    for _k in ("LLM_API_KEY", "GOOGLE_API_KEY", "LANGSMITH_API_KEY",
               "LLM_BASE_URL", "LLM_MODEL", "LLM_MODEL_SMALL"):
        try:
            os.environ[_k] = userdata.get(_k)
        except Exception:
            pass                                # el Secret no existe: se usa el default
    os.environ.setdefault("LLM_BASE_URL", "https://api.mistral.ai/v1")
    os.environ.setdefault("LLM_MODEL", "mistral-small-latest")
    os.environ.setdefault("LLM_MODEL_SMALL", "ministral-8b-latest")
except ImportError:
    from dotenv import load_dotenv             # Local: archivo .env en la raíz
    load_dotenv()

# Importar bibliotecas necesarias para memoria
from langchain_openai import ChatOpenAI
# NOTA: las clases clásicas de memoria (ConversationBufferMemory,
# ConversationSummaryMemory, ConversationBufferWindowMemory) que se describen
# más abajo están DEPRECADAS en LangChain v1. Este notebook usa el reemplazo
# oficial: RunnableWithMessageHistory. Si quieres experimentar con las clásicas:
#     from langchain_classic.memory import ConversationBufferMemory
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

import os

# --- Silenciar el aviso de deprecación de RunnableWithMessageHistory ---
# IMPORTANTE: este filtro va DESPUÉS de importar langchain, no antes. Al
# importarse, langchain_core instala su propio filtro ("default") para
# LangChainDeprecationWarning en la primera posición de warnings.filters, y
# Python aplica el primero que coincide: puesto antes, quedaría sin efecto.
#
# Seguimos usando esta API porque expone el concepto de memoria con menos piezas
# que la persistencia de LangGraph, que es su reemplazo oficial.
import warnings
warnings.filterwarnings("ignore", message=".*RunnableWithMessageHistory.*")

print("✓ Bibliotecas de memoria importadas correctamente")

✓ Bibliotecas de memoria importadas correctamente


In [3]:
# Configuración del modelo para memoria
try:
    llm = ChatOpenAI(
        base_url=os.getenv("LLM_BASE_URL"),
        api_key=os.getenv("LLM_API_KEY"),
        model=os.getenv("LLM_MODEL_SMALL", "ministral-8b-latest"),
        temperature=0.1
    )
    
    print("✓ Modelo configurado para experimentos de memoria")
    print(f"Modelo: {llm.model_name}")
    
except Exception as e:
    print(f"✗ Error en configuración: {e}")
    print("Verifica las variables de entorno")

✓ Modelo configurado para experimentos de memoria
Modelo: ministral-8b-latest


In [4]:
# Prompt con historial + entrada del usuario
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente útil."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Cadena = prompt + modelo
chain = prompt | llm

# Almacén de historiales
store = {}
def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Envolver con memoria
conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

## 1. ConversationBufferMemory - Memoria Completa

Esta memoria mantiene **todo** el historial de la conversación. Es la más simple pero puede consumir muchos tokens.

In [5]:
# Ejemplo básico con RunnableWithMessageHistory

# Prompt con historial + entrada
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente útil."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Cadena = prompt + modelo
chain = prompt | llm

# Almacén de memorias por sesión
store = {}

def get_session_history(session_id: str):
    """Devuelve (o crea) el historial completo para la sesión."""
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Envolver con RunnableWithMessageHistory
conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

def ejemplo_buffer_memory():
    print("=== CONVERSATIONBUFFERMEMORY ===")
    print("Mantiene todo el historial de conversación\n")
    
    session_id = "demo_session"

    try:
        # Primera interacción
        print("1. Primera pregunta:")
        response1 = conversation.invoke(
            {"input": "Mi nombre es Ana y soy programadora Python"},
            config={"configurable": {"session_id": session_id}}
        )
        print(f"Respuesta: {response1.content}\n")

        # Segunda interacción
        print("2. Segunda pregunta:")
        response2 = conversation.invoke(
            {"input": "¿Cuál es mi nombre y profesión?"},
            config={"configurable": {"session_id": session_id}}
        )
        print(f"Respuesta: {response2.content}\n")

        # Tercera interacción
        print("3. Tercera pregunta:")
        response3 = conversation.invoke(
            {"input": "¿Qué lenguaje de programación mencioné?"},
            config={"configurable": {"session_id": session_id}}
        )
        print(f"Respuesta: {response3.content}\n")

        # Mostrar historial
        print("=== CONTENIDO DE LA MEMORIA ===")
        history = store[session_id].messages
        for i, msg in enumerate(history, 1):
            print(f"{i}. {msg.type}: {msg.content}")

    except Exception as e:
        print(f"Error: {e}")

# Ejecutar
ejemplo_buffer_memory()


=== CONVERSATIONBUFFERMEMORY ===
Mantiene todo el historial de conversación

1. Primera pregunta:


Respuesta: ¡Hola Ana! 😊 Encantado de conocerte. Soy tu asistente virtual, aquí para ayudarte con todo lo relacionado con **Python** y más allá.

¿En qué puedo ayudarte hoy? Aquí tienes algunas ideas según tu perfil como programadora Python:

- **Preguntas sobre sintaxis o librerías** (ej: *Pandas, NumPy, Django, Flask, TensorFlow*).
- **Optimización de código** o patrones de diseño.
- **Proyectos** (desde APIs hasta automatización).
- **Buenas prácticas** (PEP 8, testing, documentación).
- **Errores específicos** (traza de código, *debugging*).
- **Recursos útiles** (libros, cursos, herramientas como VS Code, Jupyter, etc.).

O si prefieres algo más general (como aprender algo nuevo o consejos para tu carrera), ¡avísame!

**Ejemplo de cómo podrías preguntar:**
*"Ana, ¿cómo puedo hacer un scraper con BeautifulSoup para extraer datos de una web?"*
*"¿Qué librería recomiendas para analizar datos con Python?"*
*"Tengo este error en mi código de Flask: [mensaje], ¿cómo lo soluciono?"*

¡Est

Respuesta: ¡Tu nombre es **Ana** y tu profesión es **programadora Python**! 😊

Si necesitas algo más personalizado (como ajustar el tono o profundizar en algún tema), solo dime. ¡Estoy aquí para apoyarte! 🚀

*(Ejemplo: "Ana, ¿qué librerías son imprescindibles para empezar con machine learning?" o "¿Cómo puedo mejorar mi portfolio de proyectos en Python?").*

3. Tercera pregunta:


Respuesta: ¡Tú mencionaste **Python** como tu lenguaje de programación favorito (y tu profesión)!

Si quieres que profundicemos en él o en otro tema relacionado, aquí estoy. 🐍💻

¿Algo más que necesites? 😊

=== CONTENIDO DE LA MEMORIA ===
1. human: Mi nombre es Ana y soy programadora Python
2. ai: ¡Hola Ana! 😊 Encantado de conocerte. Soy tu asistente virtual, aquí para ayudarte con todo lo relacionado con **Python** y más allá.

¿En qué puedo ayudarte hoy? Aquí tienes algunas ideas según tu perfil como programadora Python:

- **Preguntas sobre sintaxis o librerías** (ej: *Pandas, NumPy, Django, Flask, TensorFlow*).
- **Optimización de código** o patrones de diseño.
- **Proyectos** (desde APIs hasta automatización).
- **Buenas prácticas** (PEP 8, testing, documentación).
- **Errores específicos** (traza de código, *debugging*).
- **Recursos útiles** (libros, cursos, herramientas como VS Code, Jupyter, etc.).

O si prefieres algo más general (como aprender algo nuevo o consejos para tu ca

## 2. ConversationBufferWindowMemory - Ventana Deslizante

Esta memoria mantiene solo los **N mensajes más recientes**, útil para controlar el uso de tokens.

In [6]:
# Prompt con historial + entrada
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente útil."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Cadena = prompt + modelo
chain = prompt | llm

# Almacén de memorias por sesión
store = {}

class WindowChatMessageHistory(BaseChatMessageHistory):
    """Historial de chat que mantiene solo los últimos k intercambios."""
    
    def __init__(self, k: int = 2):
        self.k = k
        self._messages = []
    
    @property
    def messages(self):
        # Mantener solo los últimos k intercambios (k*2 mensajes: user + assistant)
        return self._messages[-(self.k * 2):]
    
    def add_message(self, message):
        self._messages.append(message)
    
    def clear(self):
        self._messages.clear()

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    """Devuelve el historial de ventana para la sesión."""
    if session_id not in store:
        store[session_id] = WindowChatMessageHistory(k=2)
    return store[session_id]

# Envolver con RunnableWithMessageHistory
conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

# Ejemplo
def ejemplo_window_memory():
    print("=== CONVERSATION BUFFER WINDOW MEMORY (k=2) ===")
    print("Mantiene solo los 2 intercambios más recientes\n")
    
    session_id = "demo_window"
    inputs = [
        "Mi nombre es Carlos y tengo 30 años",
        "Trabajo como diseñador gráfico", 
        "Me gusta el café y la música jazz",
        "¿Puedes recordar mi edad?",
        "¿Cuál es mi profesión?"
    ]
    
    try:
        for i, user_input in enumerate(inputs, 1):
            print(f"{'='*20} INTERACCIÓN {i} {'='*20}")
            print(f"👤 Usuario: {user_input}")
            
            response = conversation.invoke(
                {"input": user_input},
                config={"configurable": {"session_id": session_id}}
            )
            print(f"🤖 Asistente: {response.content}\n")
            
            # Obtener el historial
            history = get_session_history(session_id)
            
            # Mostrar comparación clara
            total_messages = len(history._messages)
            visible_messages = len(history.messages)
            
            print(f"📊 ESTADO DE LA MEMORIA:")
            print(f"   💾 Total almacenado: {total_messages} mensajes")
            print(f"   👁️  Visible al modelo: {visible_messages} mensajes")
            print(f"   🗑️  Mensajes descartados: {total_messages - visible_messages}")
            
            # Mensajes almacenados totalmente
            print(f"\n📚 HISTORIAL COMPLETO ALMACENADO ({total_messages} mensajes):")
            if total_messages == 0:
                print("     (Ningún mensaje aún)")
            else:
                for j, msg in enumerate(history._messages, 1):
                    role = "👤 Usuario" if msg.type == "human" else "🤖 Asistente"
                    content = msg.content[:60] + "..." if len(msg.content) > 60 else msg.content
                    # Marcar si está en la ventana visible
                    is_visible = j > total_messages - visible_messages
                    marker = "✅" if is_visible else "❌"
                    print(f"     {j}. {marker} {role}: {content}")
            
            # Lo que ve el modelo
            print(f"\n🔍 VENTANA VISIBLE AL MODELO ({visible_messages} mensajes):")
            if visible_messages == 0:
                print("     (Ningún mensaje visible)")
            else:
                for j, msg in enumerate(history.messages, 1):
                    role = "👤 Usuario" if msg.type == "human" else "🤖 Asistente"
                    content = msg.content[:60] + "..." if len(msg.content) > 60 else msg.content
                    print(f"     {j}. ✅ {role}: {content}")
            
            print("\n" + "="*60 + "\n")
            
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar
ejemplo_window_memory()

=== CONVERSATION BUFFER WINDOW MEMORY (k=2) ===
Mantiene solo los 2 intercambios más recientes

==================== INTERACCIÓN 1 ====================
👤 Usuario: Mi nombre es Carlos y tengo 30 años


🤖 Asistente: ¡Hola, Carlos! Bienvenido. 😊 Soy tu asistente virtual, aquí para ayudarte con lo que necesites. ¿En qué puedo asistirte hoy? ¿Tienes alguna pregunta, necesitas información, ayuda con algo o simplemente quieres charlar? Estoy aquí para lo que requieras.

Por cierto, ¡30 años es una edad increíble! ¿Qué te gustaría hacer o saber en este momento? 🌟

📊 ESTADO DE LA MEMORIA:
   💾 Total almacenado: 2 mensajes
   👁️  Visible al modelo: 2 mensajes
   🗑️  Mensajes descartados: 0

📚 HISTORIAL COMPLETO ALMACENADO (2 mensajes):
     1. ✅ 👤 Usuario: Mi nombre es Carlos y tengo 30 años
     2. ✅ 🤖 Asistente: ¡Hola, Carlos! Bienvenido. 😊 Soy tu asistente virtual, aquí ...

🔍 VENTANA VISIBLE AL MODELO (2 mensajes):
     1. ✅ 👤 Usuario: Mi nombre es Carlos y tengo 30 años
     2. ✅ 🤖 Asistente: ¡Hola, Carlos! Bienvenido. 😊 Soy tu asistente virtual, aquí ...


==================== INTERACCIÓN 2 ====================
👤 Usuario: Trabajo como diseñador gráfico


🤖 Asistente: ¡Qué bien, Carlos! El diseño gráfico es un campo fascinante y lleno de creatividad. 🎨✨ ¿En qué área te especializas o qué tipo de proyectos sueles manejar? Por ejemplo:

- **Diseño digital** (web, apps, interfaces de usuario).
- **Branding y identidad corporativa** (logos, manuales de marca).
- **Diseño editorial** (libros, revistas, publicidad).
- **Ilustración y arte conceptual**.
- **Motion design** (animación, videos).
- **Diseño para redes sociales** (gráficos, stories, etc.).

O quizá tienes algún proyecto en mente o necesitas ayuda con algo específico, como:
- **Herramientas** (¿usas Adobe Creative Suite, Figma, Canva, Procreate, etc.?).
- **Tendencias** (¿qué estilos o técnicas te gustaría explorar?).
- **Portafolio** (¿quieres consejos para mejorarlo o destacar más?).
- **Freelance o trabajo en equipo** (¿cómo organizas tus proyectos?).
- **Inspiración** (¿buscas referencias, paletas de color o tipografías?).

¡O dime directamente qué necesitas! También puedo ayud

🤖 Asistente: ¡Carlos, me encanta que compartas tus pasiones! 🎷☕ **Café + jazz** es una combinación *perfecta* para inspirarse (y para trabajar, incluso). Aquí van algunas ideas que podrían interesarte, mezclando diseño gráfico con tu amor por el jazz y el café:

---

### **1. Inspiración visual: Jazz + Café en el diseño**
El jazz tiene un estilo **orgánico, atemporal y lleno de matices** que puede traducirse genial al diseño. Algunas referencias:
- **Paletas de color**:
  - **Tonos cálidos y profundos**: Combina el **marrón chocolate** (café) con **azules oscuros** (como el *Blue Note Records*), **verdes esmeralda** (como el *Green River Ordinary*) o **naranjas terrosos** (como el *Café de Olla*).
  - **Contraste elegante**: Prueba con **negro + blanco roto** (como los vinilos) + un toque de **rosa polvo** o **turquesa** (para dar frescura).
  - *Herramienta*: Usa [Coolors](https://coolors.co/) o [Adobe Color](https://color.adobe.com/) para crear paletas basadas en fotos de tazas de ca

🤖 Asistente: ¡No tengo memoria de tus interacciones anteriores en esta conversación! 😊 Cada vez que empiezo a ayudarte, es como si fuera la primera vez (aunque en realidad soy un modelo de IA con acceso a datos hasta 2023).

Si quieres compartir algo sobre tu edad o experiencia (por ejemplo, si eres principiante en diseño gráfico, tienes años de trayectoria, o incluso si eres de la generación que creció con el jazz en vinilos), ¡me encantaría adaptar mejor las recomendaciones! Pero no asumo nada.

**¿Qué tal si lo usamos para algo creativo?**
Por ejemplo:
- Si eres **joven y nuevo en diseño**, podríamos enfocarnos en herramientas accesibles (como Canva o Procreate) y proyectos sencillos para empezar con el estilo café-jazz.
- Si tienes **más experiencia**, podríamos explorar técnicas avanzadas (como ilustración vectorial o motion graphics para portadas de jazz).
- Si te gusta **la historia**, podríamos diseñar algo inspirado en el *jazz de los años 50* o el *café de los años 20* (como 

🤖 Asistente: ¡No tengo información sobre tu profesión en conversaciones anteriores! 😊 **No asumo nada** sobre ti (ni edad, ni trabajo, ni gustos más allá de lo que compartes aquí).

Pero si quieres que ajuste mis recomendaciones a tu contexto profesional, dime:
- **¿Eres diseñador gráfico?** (Podría sugerirte herramientas, tendencias o proyectos específicos).
- **¿Trabajas en marketing, branding o comunicación?** (Podría enfocarme en cómo aplicar el estilo café-jazz en campañas).
- **¿Eres músico, barista o estudias algo relacionado?** (Podría conectar tus pasiones con el diseño de una manera más personal).
- **¿O simplemente te gusta el café y el jazz como hobby?** (¡También está genial! Podría recomendarte proyectos relajados o para tu tiempo libre).

---
### **Ejemplo de cómo adaptaría las recomendaciones según tu profesión** (por si te sirve de inspiración):
| **Profesión**       | **Enfoque en café + jazz**                                                                 | **Ejempl

## 3. ConversationSummaryMemory - Resumen Inteligente

Esta memoria **resume** conversaciones largas en lugar de mantener todo el texto completo, ahorrando tokens significativamente.

In [7]:

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Función para resumir automáticamente cuando hay muchos mensajes
def auto_summarize(session_id: str, max_messages=6):
    history = get_session_history(session_id)
    
    if len(history.messages) > max_messages:
        # Mensajes a resumir (todos excepto los últimos 2)
        messages_to_summarize = history.messages[:-2]
        
        # Crear texto para resumir
        conversation_text = ""
        for msg in messages_to_summarize:
            role = "Usuario" if msg.type == "human" else "Asistente"
            conversation_text += f"{role}: {msg.content}\n"
        
        # Generar resumen
        summary_response = llm.invoke(f"Resume esta conversación en 2-3 líneas:\n{conversation_text}")
        summary = summary_response.content
        
        # Reemplazar mensajes antiguos con el resumen
        recent_messages = history.messages[-2:]
        history.clear()
        history.add_ai_message(f"[RESUMEN]: {summary}")
        history.messages.extend(recent_messages)

# Crear conversación
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente útil."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

conversation = RunnableWithMessageHistory(
    prompt | llm,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

def ejemplo_summary_memory():
    print("=== CONVERSATION SUMMARY MEMORY ===")
    print("Resume conversaciones largas para ahorrar tokens\n")
    
    session_id = "summary_session"
    
    # Conversación de ejemplo
    inputs = [
        "Hola, soy María González, ingeniera de software de 35 años",
        "Trabajo en una startup de fintech en Madrid desarrollando pagos digitales",
        "Usamos React, Node.js, Docker y Kubernetes en nuestros proyectos",
        "Mi mayor desafío es la latencia en transacciones internacionales",
        "También trabajo en mejorar la UX de nuestra app móvil",
        "¿Puedes resumir quién soy y cuáles son mis principales desafíos?"
    ]
    
    try:
        for i, user_input in enumerate(inputs, 1):
            print(f"{'='*15} INTERACCIÓN {i} {'='*15}")
            print(f"👤 Usuario: {user_input}")
            
            # Resumir automáticamente si es necesario
            auto_summarize(session_id)
            
            response = conversation.invoke(
                {"input": user_input},
                config={"configurable": {"session_id": session_id}}
            )
            print(f"🤖 Asistente: {response.content}\n")
            
            # Mostrar estado de la memoria
            history = get_session_history(session_id)
            total_messages = len(history.messages)
            
            print(f"📊 ESTADO DE LA MEMORIA:")
            print(f"   💾 Total mensajes: {total_messages}")
            
            # Verificar si hay resumen
            has_summary = any("[RESUMEN]" in msg.content for msg in history.messages if hasattr(msg, 'content'))
            print(f"   📝 Tiene resumen: {'✅ Sí' if has_summary else '❌ No'}")
            
            print(f"\n💬 CONTENIDO ACTUAL DE LA MEMORIA:")
            for j, msg in enumerate(history.messages, 1):
                role = "👤 Usuario" if msg.type == "human" else "🤖 Asistente"
                content = msg.content
                
                # Destacar si es un resumen
                if "[RESUMEN]" in content:
                    role = "📝 Resumen"
                    content = content.replace("[RESUMEN]: ", "")
                
                # Truncar si es muy largo
                if len(content) > 80:
                    content = content[:80] + "..."
                
                print(f"   {j}. {role}: {content}")
            
            print("\n" + "="*50 + "\n")
            
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar
ejemplo_summary_memory()

=== CONVERSATION SUMMARY MEMORY ===
Resume conversaciones largas para ahorrar tokens

=============== INTERACCIÓN 1 ===============
👤 Usuario: Hola, soy María González, ingeniera de software de 35 años


🤖 Asistente: ¡Hola, María! Encantado de conocerte. 😊 Como ingeniera de software con esa experiencia y perspectiva, seguro que tienes un perfil muy interesante. ¿En qué puedo ayudarte hoy? Aquí van algunas ideas según lo que pueda necesitar:

1. **Desarrollo profesional**:
   - ¿Buscas consejos para crecer en tu carrera (ej.: liderazgo técnico, mentoría, especialización en áreas como *cloud*, *IA*, *DevOps*, etc.)?
   - ¿Te interesa cómo equilibrar proyectos técnicos con habilidades blandas (comunicación, gestión de equipos)?

2. **Tecnología y herramientas**:
   - ¿Quieres profundizar en algún lenguaje/framework (Python, Java, Kubernetes, etc.) o explorar tendencias como *low-code*, *serverless* o *cyberseguridad*?
   - ¿Necesitas ayuda con algo específico de tu código o arquitectura?

3. **Mujeres en tech**:
   - ¿Te gustaría compartir experiencias sobre cómo es ser mujer en la industria? O quizá recursos para inspirar a otras ingenieras (ej.: comunidades como *Women Who Code*, *Girls

🤖 Asistente: ¡Qué emocionante! Trabajar en una **fintech** en Madrid es un entorno **dinámico, desafiante y lleno de oportunidades** —especialmente en pagos digitales, donde la innovación y la regulación (como el **PSD2** o los **open banking**) son clave. 💡💳

Dado tu perfil y contexto, aquí tienes ideas **personalizadas** para explorar, según lo que pueda interesarte. Dime si quieres profundizar en alguna o ajustar el enfoque:

---

### **1. Áreas técnicas clave en pagos digitales (y cómo destacarte)**
Tu trabajo probablemente toca temas como:
- **Integraciones bancarias**:
  - ¿Has trabajado con **APIs de bancos** (ej.: *Sabadell, BBVA, Revolut*) o proveedores como **Adyen, Stripe, o Mercado Pago**?
  - *Oportunidad*: Aprender sobre **open banking** (normativa **PSD2** en Europa) o **tokenización** (para mayor seguridad en pagos).
  - *Recurso*: Documentación de la [EBA (Autoridad Bancaria Europea)](https://www.eba.europa.eu/) o cursos en [Coursera sobre finanzas digitales](https://w

🤖 Asistente: ¡Perfecto, María! Con tu stack actual (**React + Node.js + Docker + Kubernetes**) en una **fintech de pagos digitales**, tienes un perfil **muy sólido y versátil**, especialmente en un entorno donde la **escalabilidad, seguridad y rendimiento** son críticos. Vamos a profundizar en cómo **optimizar tu trabajo, escalar soluciones y destacar** en este contexto, con ejemplos prácticos y recursos específicos para tu caso.

---

## **1. Optimización del Stack Actual**
### **A. Backend (Node.js + Kubernetes)**
#### **1.1. Escalabilidad para pagos en tiempo real**
En fintech, los pagos requieren:
- **Baja latencia** (ej.: procesar una transacción en <500ms).
- **Alta disponibilidad** (ej.: tolerancia a fallos en APIs bancarias).
- **Escalado horizontal** (ej.: manejar picos de tráfico como el **Black Friday**).

**Acciones concretas:**
- **Kubernetes optimizado para Node.js**:
  - Usa **Horizontal Pod Autoscaler (HPA)** con métricas personalizadas (ej.: escalar según la cola de pa

🤖 Asistente: ¡Entiendo perfectamente, María! **La latencia en transacciones internacionales** es uno de los mayores dolores de cabeza en fintech, especialmente cuando trabajas con pagos digitales. Los usuarios esperan **respuestas en segundos** (o incluso milisegundos), pero factores como:
- **Integraciones con bancos internacionales** (APIs lentas o con TPS limitados).
- **Conversión de divisas en tiempo real**.
- **Fronteras geográficas** (latencia de red entre regiones).
- **Cumplimiento regulatorio** (ej.: verificación KYC/AML en tiempo real).

Vamos a **desglosar soluciones técnicas y arquitecturales** basadas en tu stack (**React + Node.js + Docker + Kubernetes**), priorizando **baja latencia, escalabilidad y resiliencia**. Incluiré ejemplos concretos, métricas y herramientas que ya usas (o podrías adoptar).

---

## **🔍 Análisis del Problema**
### **Causas comunes de latencia en pagos internacionales**
| Causa                          | Ejemplo                                  |

🤖 Asistente: ¡Excelente que estés trabajando en **UX de la app móvil**, María! En fintech, una **buena experiencia de usuario (UX)** puede marcar la diferencia entre que un cliente complete una transacción en **<3 segundos** o abandone la app por frustración. Dado que trabajas con **React (frontend) + Node.js (backend)**, podemos optimizar la UX desde **diseño, interacciones, rendimiento y feedback en tiempo real**, combinando técnicas de **UX/UI, psicología del usuario y optimización técnica**.

---
## **🎯 Objetivos Clave para la UX de Pagos Móviles**
1. **Reducir fricción** en el flujo de pago (menos pasos, menos errores).
2. **Minimizar la percepción de latencia** (ej.: skeleton loaders, micro-interacciones).
3. **Validar datos en tiempo real** (evitar errores al final del proceso).
4. **Personalizar la experiencia** (ej.: recordatorios de transacciones frecuentes).
5. **Incluir feedback claro** (ej.: notificaciones de éxito/fracaso con acciones claras).

---
## **🔍 Análisis de Pain

🤖 Asistente: **📌 *Resumen ejecutivo de tu perfil y desafíos* (en 3 líneas clave + contexto):**

---
**👩‍💻 *Quién eres*:**
Eres **desarrolladora full-stack especializada en fintech** (React + Node.js + Docker/Kubernetes) en una *startup de pagos digitales* en Madrid, donde **escalabilidad, seguridad y rendimiento** son críticos para transacciones internacionales. Tu stack técnico combina **arquitecturas distribuidas** (microservicios, APIs PCI DSS-compliant) con **experiencia en optimización de sistemas** para manejar picos de tráfico, pero buscas **impacto en UX y crecimiento profesional** en un sector competitivo.

---
**🔥 *Tus 3 desafíos principales* (priorizados):**
1. **Latencia en transacciones internacionales** (ej.: APIs bancarias lentas, conversiones de divisas, latencia de red entre regiones).
   - *Impacto*: Usuarios abandonan flujos si la respuesta supera **500ms** (ej.: fallos en pagos a EE.UU. o Asia).
   - *Dato clave*: La latencia puede aumentar un **30-50%** por integra

## Consideraciones Técnicas y Mejores Prácticas

### Selección del Tipo de Memoria

| Tipo | Cuándo Usarlo | Ventajas | Desventajas |
|------|---------------|----------|-------------|
| **Buffer** | Conversaciones cortas | Contexto completo | Alto consumo de tokens |
| **Window** | Contexto reciente importante | Eficiente en tokens | Puede perder información clave |
| **Summary** | Conversaciones muy largas | Balance eficiencia/contexto | Pérdida de detalles específicos |

### Mejores Prácticas:

1. **Gestión de Tokens**:
   - Monitorea el uso de tokens regularmente
   - Establece límites máximos para evitar costos excesivos
   - Considera el costo vs. calidad del contexto

2. **Selección Estratégica**:
   - Usa Buffer para sesiones cortas e importantes
   - Usa Window para conversaciones con contexto limitado
   - Usa Summary para sesiones largas de asistencia

3. **Optimización**:
   - Limpia memoria periódicamente si es necesario
   - Implementa estrategias híbridas según el caso de uso
   - Considera almacenamiento persistente para memoria a largo plazo

## Ejercicios Prácticos

### Ejercicio 1: Análisis de Consumo
Implementa un sistema que monitoree y reporte el uso de tokens con diferentes tipos de memoria.

### Ejercicio 2: Memoria Híbrida
Diseña una estrategia que combine multiple tipos de memoria según el contexto.

### Ejercicio 3: Persistencia
Extiende el chatbot para guardar y cargar memoria entre sesiones.

## Conceptos Clave Aprendidos

1. **Importancia de la memoria** en conversaciones naturales
2. **Tipos de memoria** y sus casos de uso específicos
3. **Balance** entre contexto y eficiencia de tokens
4. **Implementación práctica** con LangChain
5. **Estrategias de optimización** para diferentes escenarios

## Conclusión del Módulo IL1.1

Has completado la introducción a LLMs y conexiones API. Los conceptos aprendidos:

1. **APIs directas** vs **frameworks** como LangChain
2. **Streaming** para mejor experiencia de usuario
3. **Memoria** para conversaciones contextuales
4. **Mejores prácticas** de seguridad y optimización

### Próximos Pasos
En **IL1.2** exploraremos técnicas avanzadas de **prompt engineering** incluyendo zero-shot, few-shot, y chain-of-thought prompting.